<a href="https://colab.research.google.com/github/Manpreet102005/Airline-Flight-Price-Predictor/blob/main/Airline_Flight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.express as px


In [2]:
df=pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Airline-Flight/airlines_flights_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Airline-Flight/airlines_flights_data.csv'

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# class distribution
plt.pie(df['class'].value_counts(),labels=df['class'].value_counts().index,)
plt.title("Class Distribution")
plt.show()

In [ ]:
#airline distribution
plt.pie(df['airline'].value_counts(),labels=df['airline'].value_counts().index)
plt.title("Airline Distribution")
plt.show()

In [ ]:
source_df=df.source_city.value_counts()
dest_df=df.destination_city.value_counts()

In [ ]:
# No of flights from particular city
px.bar(source_df,x=source_df.index,y=source_df.values,title='No of flights from particular city',labels={'x':'City','y':'No of flights'})

In [ ]:
# No of flights from particular city
px.bar(dest_df,x=dest_df.index,y=dest_df.values,title='No of flights to particular city',labels={'x':'City','y':'No of flights'})

In [ ]:
# count of stops
px.bar(df['stops'].value_counts(),x=df['stops'].value_counts().index,y=df['stops'].value_counts().values,labels={'x':'Stops','y':'No of flights'},title='Count of stops')

In [ ]:
px.bar(df['departure_time'].value_counts(),
       x=df['departure_time'].value_counts().index,
       y=df['departure_time'].value_counts().values,
       labels={'x':'Departure Time','y':'Number of Flights'},
       title='Number of Flights by Departure Time')

In [ ]:
#scatter matrix
px.scatter_matrix(df,dimensions=['duration','days_left','price'])

In [ ]:
#checking duplicate entries
df.duplicated().sum()

In [ ]:
df.drop(columns=['index','flight'],axis=1,inplace=True)

In [ ]:
px.histogram(df,x='price',title='Price Distribution',labels={'price':'Price'})

In [ ]:
px.histogram(df,x='duration',title='Duration Distribution',labels={'duration':'Duration'})

In [ ]:
px.histogram(df,x='days_left',title='Days Left Distribution',labels={'days_left':'Days Left'})

In [ ]:
# Outlier Handling- Price and Days left
Q1_price = df['price'].quantile(0.25)
Q3_price = df['price'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound_price = Q1_price - 1.5 * IQR_price
upper_bound_price = Q3_price + 1.5 * IQR_price

Q1_duration = df['duration'].quantile(0.25)
Q3_duration = df['duration'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = Q1_duration - 1.5 * IQR_duration
upper_bound_duration = Q3_duration + 1.5 * IQR_duration

df = df[(df['price'] >= lower_bound_price) & (df['price'] <= upper_bound_price)]
df = df[(df['duration'] >= lower_bound_duration) & (df['duration'] <= upper_bound_duration)]

In [ ]:
df.describe()

In [ ]:
px.histogram(df,x='duration',title='Duration Distribution After Removing Outliers',labels={'duration':'Duration'})

In [ ]:
px.histogram(df,x='price',title='Price Distribution After Removing Outliers',labels={'price':'Price'})

In [ ]:
#log transformation Tranformation -price,duration
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
transformer=ColumnTransformer(transformers=[('lg',FunctionTransformer(np.log1p),['duration'])])
df[['duration']]=transformer.fit_transform(df[['duration']])

In [ ]:
price_transformer=ColumnTransformer(transformers=[('lg',FunctionTransformer(np.log1p),['price'])])
df[['price']]=price_transformer.fit_transform(df[['price']])

In [ ]:
import joblib
joblib.dump(transformer,'transformer.pkl')

In [ ]:
px.histogram(df,x='duration',title='Duration Distribution After Outlier Handling and Transformation ',labels={'duration':'Duration'})

In [ ]:
px.histogram(df,x='price',title='Duration Distribution After Outlier handling and Transformation',labels={'price':'Price'})

In [ ]:
#splitting
from sklearn.model_selection import train_test_split
train,test_df=train_test_split(df,test_size=0.2,random_state=123)
train_df,val_df=train_test_split(train,random_state=123,test_size=0.25)

In [ ]:
train_df

In [ ]:
#selecting inputs
input_cols=list(train_df.columns[:-1])
target_col='price'

In [ ]:
input_cols,target_col

In [ ]:
x_train=train_df[input_cols].copy()
x_test=test_df[input_cols].copy()
x_val=val_df[input_cols].copy()


In [ ]:
Y_train=train_df[target_col].copy()
Y_test=test_df[target_col].copy()
Y_val=val_df[target_col].copy()

In [ ]:
#numeric_cols
numeric_cols=list(x_train.select_dtypes('number'))
cat_cols=list(x_train.select_dtypes('object'))

In [ ]:
numeric_cols,cat_cols

In [ ]:
# scaling numeric_cols
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler().fit(x_train[numeric_cols])
x_train[numeric_cols]=scaler.transform(x_train[numeric_cols])
x_test[numeric_cols]=scaler.transform(x_test[numeric_cols])
x_val[numeric_cols]=scaler.transform(x_val[numeric_cols])


In [ ]:
joblib.dump(scaler,'scaler.pkl')

In [ ]:
# one hot encoding
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(handle_unknown='ignore').fit(x_train[cat_cols])
encoded_cols=encoder.get_feature_names_out(cat_cols)
encoded_cols

In [ ]:
joblib.dump(encoder,'encoder.pkl')

In [ ]:
x_train[encoded_cols]=encoder.transform(x_train[cat_cols]).toarray()
x_test[encoded_cols]=encoder.transform(x_test[cat_cols]).toarray()
x_val[encoded_cols]=encoder.transform(x_val[cat_cols]).toarray()


In [ ]:
x_train.drop(columns=cat_cols,inplace=True)
x_test.drop(columns=cat_cols,inplace=True)
x_val.drop(columns=cat_cols,inplace=True)

In [ ]:
all_cols = numeric_cols + list(encoded_cols)

X_train = x_train[all_cols].copy()
X_test = x_test[all_cols].copy()
X_val = x_val[all_cols].copy()

In [ ]:
X_train.info()

Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
linear_regression=LinearRegression(n_jobs=-1).fit(X_train,Y_train)

In [ ]:
def check_rmse(model):
  from sklearn.metrics import root_mean_squared_error
  pred_t,pred_v=model.predict(X_train),model.predict(X_val)
  print(f"Model: {model}")
  print(f"Train RMSE: {root_mean_squared_error(np.expm1(pred_t),np.expm1(Y_train))}")
  print(f"Validation RMSE: {root_mean_squared_error(np.expm1(pred_v),np.expm1(Y_val))}")

In [ ]:
check_rmse(linear_regression)

Lasso Model

In [ ]:
from sklearn.linear_model import Lasso
lasso=Lasso().fit(X_train,Y_train)

In [ ]:
check_rmse(lasso)

In [ ]:
lasso=Lasso(alpha=1.5,max_iter=2000,tol=1e-06,random_state=123,warm_start=True,precompute=True).fit(X_train,Y_train)

In [ ]:
check_rmse(lasso)

Lasso is same as Linear Regression even after hyperparameter tuning

Ridge Model

In [ ]:
from sklearn.linear_model import Ridge
ridge=Ridge(alpha=0.5,max_iter=1500,tol=1e-6,random_state=123).fit(X_train,Y_train)

In [ ]:
check_rmse(ridge)

SGD Regressor

In [ ]:
from sklearn.linear_model import SGDRegressor
sgd=SGDRegressor(random_state=123).fit(X_train,Y_train)
check_rmse(sgd)



In [ ]:
space={
    'alpha':[0.0001,0.001,0.01,0.1,1,10],
    'penalty':['l2','l1','elasticnet'],
    'max_iter':[1000,1500,2000],
    'tol':[1e-3,1e-4,1e-5,1e-6],
    'learning_rate':['constant','optimal','invscaling','adaptive'],
}

In [ ]:
random_search_cv(sgd,space)

In [ ]:
np.sqrt(45736041.8866356)

Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor
decision_tree=DecisionTreeRegressor(random_state=123).fit(X_train,Y_train)

In [ ]:
check_rmse(decision_tree)

Grid Search and Random Search CV

In [ ]:
def grid_search_cv(model,space):
  from sklearn.model_selection import GridSearchCV, KFold
  cv=KFold(n_splits=7,shuffle=True,random_state=123)
  model = GridSearchCV(model, space, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
  model.fit(X_train,Y_train)
  print(f"Best Params: {model.best_params_}")
  print(f"Best Scores: {model.best_score_}")
  return model

def random_search_cv(model,space):
  from sklearn.model_selection import RandomizedSearchCV, KFold
  cv=KFold(n_splits=7,shuffle=True,random_state=123)
  model = RandomizedSearchCV(model, space, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
  model.fit(X_train,Y_train)
  print(f"Best Params: {model.best_params_}")
  print(f"Best Scores: {model.best_score_}")


In [ ]:
DecisionTreeRegressor?

In [ ]:
def check_rmse_2(model):
  from sklearn.metrics import root_mean_squared_error
  pred_t,pred_v=model.predict(X_train),model.predict(X_val)
  train_error=root_mean_squared_error(np.expm1(pred_t),np.expm1(Y_train))
  val_error=root_mean_squared_error(np.expm1(pred_v),np.expm1(Y_val))
  return train_error,val_error

T=list()
V=list()
splits = list(range(20, 40))
for i in splits:
  decision_tree=DecisionTreeRegressor(random_state=123,max_depth=22,max_features='sqrt',min_samples_leaf=1,min_samples_split=i,splitter='best').fit(X_train,Y_train)
  train_error,val_error=check_rmse_2(decision_tree)
  T.append(train_error)
  V.append(val_error)

plt.plot(splits, T, label='Train error')
plt.plot(splits, V, label='Validation error')
plt.xlabel('Max Depth')
plt.ylabel('RMSE')
plt.title('RMSE vs. Max Split for Decision Tree')
plt.legend()
plt.show()

In [ ]:
decision_tree=DecisionTreeRegressor(random_state=123,max_depth=30,max_features='sqrt',min_samples_leaf=1,min_samples_split=24,splitter='best').fit(X_train,Y_train)

In [ ]:
check_rmse(decision_tree)

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
random_forest=RandomForestRegressor(random_state=123,n_jobs=-1).fit(X_train,Y_train)

In [ ]:
check_rmse(random_forest)

In [ ]:
space={
    'n_estimators':[100,150,200],
    'max_depth':[20,22,25,28,30],
    'max_features':['sqrt','log2'],
    'max_leaf_nodes':[3,4,8],
    'min_samples_split':[2,4,6]
}
random_search_cv(random_forest,space)

In [ ]:
np.sqrt(83024526)

In [ ]:
random_forest=RandomForestRegressor(random_state=123,n_jobs=-1,n_estimators=200,max_depth=50,max_features='sqrt',max_leaf_nodes=32,min_samples_split=35).fit(X_train,Y_train)

In [ ]:
check_rmse(random_forest)

XGBoost

In [ ]:
from xgboost import XGBRegressor
xgboost=XGBRegressor(random_state=123,n_jobs=-1).fit(X_train,Y_train)

In [ ]:
check_rmse(xgboost)

In [ ]:
space={
    'n_estimators':[100,150,200],
    'max_depth':[5,6,7,8,9,10],

}
random_search_cv(xgboost,space)

In [ ]:
np.sqrt(15777755.0)

In [ ]:
xgboost=XGBRegressor(random_state=123,n_jobs=-1,n_estimators=300,max_depth=9).fit(X_train,Y_train)

In [ ]:
check_rmse(xgboost)

In [ ]:
random_forest=RandomForestRegressor(random_state=123,n_jobs=-1).fit(X_train,Y_train)
xgboost=XGBRegressor(random_state=123,n_jobs=-1,n_estimators=300,max_depth=9).fit(X_train,Y_train)

In [ ]:
Y_train_back_transfored=np.expm1(Y_train)
Y_val_back_transfored=np.expm1(Y_val)

In [ ]:
Y_train_back_transfored.describe()

In [ ]:
np.divide(1826,np.subtract(98972,1105))*100

**Error % on Training Set: 1.86**


In [ ]:
Y_val_back_transfored.describe()

In [ ]:
np.divide(2591,np.subtract(98904,1105))*100

**Error % on Validation Set: 2.64**


# Stacking And Voting
 on these two best models:


1.   RandomForest

    Train RMSE: 1148.4745139251934
    Validation RMSE: 2713.465109612326


---


2.   XGBRegressor

    Train RMSE: 1826.5382796119432
    Validation RMSE: 2591.599767634328



In [ ]:
from sklearn.ensemble import StackingRegressor
stacked=StackingRegressor(n_jobs=-1,estimators=[('rf',random_forest),('xgb',xgboost)],final_estimator=LinearRegression()).fit(X_train,Y_train)

In [ ]:
check_rmse(stacked)

In [ ]:
from sklearn.ensemble import VotingRegressor
voting=VotingRegressor(n_jobs=-1,estimators=[('rf',random_forest),('xgb',xgboost)]).fit(X_train,Y_train)


In [ ]:
check_rmse(voting)

# Final Model: Voting Regressor

Train RMSE: 1405.6261651075552

Validation RMSE: 2514.576838985678

In [ ]:
import joblib
joblib.dump(stacked,'airline-price-predictor.pkl')

In [ ]:
joblib.dump(xgboost,'xgboost_model.pkl')